# Post-May-25 — QRC Anchor-Snapshot Diagnostics

This notebook compares the May-25 final-state TFIM-QRC readout against the first working-prototype upgrade: anchor-snapshot / virtual-node observable collection.

Scope:

- target: `future_rv_20d`;
- compare 6q/PCA-6 and 8q/PCA-8;
- compare final-state features vs anchor-snapshot features;
- use `Z+X+ZZ` observables;
- inspect RMSE, QLIKE, Mincer-Zarnowitz, and reservoir feature diagnostics.

This is not a large hyperparameter sweep. The goal is to decide whether anchor snapshots produce more informative reservoir features before tuning reservoir dynamics.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Load data and build PCA sequence windows

In [2]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

pca8 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=8,
    prefix="pca8",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

sequence_splits_8 = make_qrc_sequence_splits(
    pca8.splits,
    feature_columns=pca8.feature_columns,
    target_column=target,
    lookback_days=40,
)

In [6]:
print("PCA-6 cumulative variance:", pca6.explained_variance["cumulative_explained_variance"].iloc[-1])
print("PCA-8 cumulative variance:", pca8.explained_variance["cumulative_explained_variance"].iloc[-1])

print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_6.items()})
print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_8.items()})

PCA-6 cumulative variance: 0.8048920431047643
PCA-8 cumulative variance: 0.8740976542943539
{'train': ((5420, 40, 6), (5420,)), 'val': ((1219, 40, 6), (1219,)), 'test': ((1019, 40, 6), (1019,))}
{'train': ((5420, 40, 8), (5420,)), 'val': ((1219, 40, 8), (1219,)), 'test': ((1019, 40, 8), (1019,))}


## 2. Define four comparison cases

In [7]:
runs = [
    {
        "run_name": "6q_pca6_zxzz_final",
        "sequence_splits": sequence_splits_6,
        "config": TFIMQRCConfig(
            qubits=6,
            pca_components=6,
            lookback_days=40,
            anchor_count=6,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=False,
        ),
    },
    {
        "run_name": "6q_pca6_zxzz_anchor_snapshots",
        "sequence_splits": sequence_splits_6,
        "config": TFIMQRCConfig(
            qubits=6,
            pca_components=6,
            lookback_days=40,
            anchor_count=6,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=True,
        ),
    },
    {
        "run_name": "8q_pca8_zxzz_final",
        "sequence_splits": sequence_splits_8,
        "config": TFIMQRCConfig(
            qubits=8,
            pca_components=8,
            lookback_days=40,
            anchor_count=8,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=False,
        ),
    },
    {
        "run_name": "8q_pca8_zxzz_anchor_snapshots",
        "sequence_splits": sequence_splits_8,
        "config": TFIMQRCConfig(
            qubits=8,
            pca_components=8,
            lookback_days=40,
            anchor_count=8,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=True,
        ),
    },
]

## 3. Run QRC comparisons

In [8]:
result_rows = []
diagnostic_rows = []
results = {}

for spec in runs:
    run_name = spec["run_name"]
    config = spec["config"]
    sequence_splits = spec["sequence_splits"]

    print(f"Running {run_name}")
    result = fit_tfim_qrc_regressor(
        sequence_splits,
        config=config,
        target=target,
        verbose=True,
    )
    results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    result_rows.append(row)

    _, y_train, _ = sequence_splits["train"]
    _, y_val, _ = sequence_splits["val"]
    _, y_test, _ = sequence_splits["test"]

    diag = diagnose_reservoir_feature_splits(
        result.train_features,
        result.val_features,
        result.test_features,
        y_train,
        y_val,
        y_test,
    )
    diag.insert(0, "run_name", run_name)
    diagnostic_rows.append(diag)

result_table = pd.DataFrame(result_rows)
diagnostics_table = pd.concat(diagnostic_rows, ignore_index=True)

Running 6q_pca6_zxzz_final
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running 6q_pca6_zxzz_anchor_snapshots
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 325

## 4. Metrics table

In [9]:
metric_cols = [
    "run_name",
    "qubits",
    "pca_components",
    "anchor_count",
    "observable_mode",
    "collect_anchor_features",
    "n_reservoir_features",
    "train_rmse",
    "val_rmse",
    "test_rmse",
    "train_qlike",
    "val_qlike",
    "test_qlike",
    "train_mz_r2",
    "val_mz_r2",
    "test_mz_r2",
]

result_table[metric_cols].sort_values("test_rmse")

,run_name,qubits,pca_components,anchor_count,observable_mode,collect_anchor_features,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,6q_pca6_zxzz_final,6,6,6,zxzz,False,17,0.094534,0.069317,0.105705,-2.270258,-2.961464,-1.961164,0.124484,0.017802,0.038175
1,6q_pca6_zxzz_anchor_snapshots,6,6,6,zxzz,True,102,0.082319,0.066934,0.106517,-2.507062,-2.951327,-1.850841,0.330713,0.022583,0.045200
3,8q_pca8_zxzz_anchor_snapshots,8,8,8,zxzz,True,184,0.081400,0.067124,0.108023,-2.514880,-2.919680,-1.756914,0.347181,0.022811,0.035752
2,8q_pca8_zxzz_final,8,8,8,zxzz,False,23,0.097706,0.068201,0.109260,-2.194921,-2.983476,-1.852447,0.068716,0.006665,0.008340


## 5. Reservoir feature diagnostics

In [10]:
diagnostic_cols = [
    "run_name",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

diagnostics_table[diagnostic_cols]

,run_name,split,n_samples,n_features,near_constant_features,feature_std_min,feature_std_median,feature_std_max,effective_rank,condition_number,mean_abs_feature_target_corr,max_abs_feature_target_corr,mean_abs_shift_vs_train,max_abs_shift_vs_train
0,6q_pca6_zxzz_final,train,5420,17,0,0.174050,0.228300,0.331968,16.215538,3.087385,0.112299,0.226706,0.000000,0.000000
1,6q_pca6_zxzz_final,val,1219,17,0,0.151658,0.215078,0.337574,15.955161,3.707712,0.071243,0.185666,0.133893,0.285595
2,6q_pca6_zxzz_final,test,1019,17,0,0.162652,0.222064,0.302210,16.182298,3.455584,0.084340,0.190785,0.118247,0.222432
3,6q_pca6_zxzz_anchor_snapshots,train,5420,102,0,0.174050,0.273022,0.564309,79.883487,33.779875,0.120302,0.467016,0.000000,0.000000
4,6q_pca6_zxzz_anchor_snapshots,val,1219,102,0,0.127679,0.252789,0.538400,75.111205,58.281116,0.057964,0.198933,0.162537,0.904979
5,6q_pca6_zxzz_anchor_snapshots,test,1019,102,0,0.162652,0.266522,0.543450,78.259309,39.675438,0.080315,0.225915,0.172536,0.600421
6,8q_pca8_zxzz_final,train,5420,23,0,0.126733,0.177647,0.319163,21.547459,3.889368,0.084515,0.145142,0.000000,0.000000
7,8q_pca8_zxzz_final,val,1219,23,0,0.121140,0.183618,0.320065,21.265580,4.987527,0.046811,0.123117,0.103736,0.281853
8,8q_pca8_zxzz_final,test,1019,23,0,0.118123,0.178249,0.286299,21.430775,4.482487,0.074946,0.203200,0.083846,0.256754
9,8q_pca8_zxzz_anchor_snapshots,train,5420,184,0,0.120893,0.230643,0.564309,139.401477,55.209729,0.095974,0.467016,0.000000,0.000000


## 6. Save results

In [11]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

result_table.to_csv(out_dir / "phase2_qrc_anchor_snapshot_comparison.csv", index=False)
diagnostics_table.to_csv(out_dir / "phase2_qrc_anchor_snapshot_diagnostics.csv", index=False)

print("Saved comparison and diagnostics to", out_dir)

Saved comparison and diagnostics to results/tables


## 7. Interpretation rule

Anchor snapshots justify further tuning only if they improve at least one of:

```text
test MZ R²
test QLIKE
feature-target correlation
effective rank without severe train/test shift
```

If snapshots do not improve these diagnostics, the next step is not a parameter sweep; it is heterogeneous fixed TFIM dynamics or a residual readout design.

In [12]:
# Ridge-alpha sweep for 6q / PCA-6 / ZXZZ / anchor snapshots

ridge_alphas = [10, 30, 100, 300, 1000, 3000]

ridge_rows = []
ridge_results = {}

for alpha in ridge_alphas:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode="zxzz",
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=float(alpha),
        target_transform="log",
        seed=42,
        collect_anchor_features=True,
    )

    run_name = f"6q_pca6_zxzz_anchor_snapshots_ridge_{alpha}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    ridge_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    ridge_rows.append(row)

ridge_sweep = pd.DataFrame(ridge_rows)

ridge_sweep[
    [
        "run_name",
        "ridge_alpha",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
].sort_values("test_rmse")

Running 6q_pca6_zxzz_anchor_snapshots_ridge_10
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running 6q_pca6_zxzz_anchor_snapshots_ridge_30
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sa

,run_name,ridge_alpha,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
5,6q_pca6_zxzz_anchor_snapshots_ridge_3000,3000.0,102,0.084864,0.065424,0.103646,-2.478240,-2.973016,-1.883750,0.309251,0.023966,0.055548
4,6q_pca6_zxzz_anchor_snapshots_ridge_1000,1000.0,102,0.083484,0.065886,0.104204,-2.495224,-2.963891,-1.871821,0.321554,0.023920,0.053969
3,6q_pca6_zxzz_anchor_snapshots_ridge_300,300.0,102,0.082752,0.066325,0.105019,-2.503095,-2.957899,-1.864544,0.327778,0.023427,0.050743
2,6q_pca6_zxzz_anchor_snapshots_ridge_100,100.0,102,0.082469,0.066624,0.105690,-2.505881,-2.954423,-1.858340,0.329968,0.023064,0.048147
1,6q_pca6_zxzz_anchor_snapshots_ridge_30,30.0,102,0.082351,0.066833,0.106233,-2.506871,-2.952260,-1.853230,0.330666,0.022769,0.046175
0,6q_pca6_zxzz_anchor_snapshots_ridge_10,10.0,102,0.082319,0.066934,0.106517,-2.507062,-2.951327,-1.850841,0.330713,0.022583,0.045200


In [13]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

ridge_sweep.to_csv(
    out_dir / "phase2_qrc_anchor_snapshot_ridge_sweep.csv",
    index=False,
)

print("Saved ridge sweep to", out_dir)

Saved ridge sweep to results/tables


In [3]:
# Fixed-disorder probe:
# 6q / PCA-6 / ZXZZ / anchor snapshots / ridge_alpha=3000

disorder_strengths = [0.05, 0.10, 0.20]

disorder_rows = []
disorder_results = {}

for strength in disorder_strengths:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode="zxzz",
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        collect_anchor_features=True,
        use_disorder=True,
        disorder_strength=float(strength),
    )

    run_name = f"6q_pca6_zxzz_snapshots_alpha3000_disorder_{strength}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    disorder_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    disorder_rows.append(row)

disorder_sweep = pd.DataFrame(disorder_rows)

disorder_sweep[
    [
        "run_name",
        "ridge_alpha",
        "use_disorder",
        "disorder_strength",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
].sort_values("test_rmse")

Running 6q_pca6_zxzz_snapshots_alpha3000_disorder_0.05
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running 6q_pca6_zxzz_snapshots_alpha3000_disorder_0.1
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2

,run_name,ridge_alpha,use_disorder,disorder_strength,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
2,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.2,3000.0,True,0.20,102,0.084320,0.063489,0.102618,-2.485080,-2.996562,-1.942716,0.319206,0.039238,0.072134
1,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.1,3000.0,True,0.10,102,0.084465,0.064642,0.103050,-2.483934,-2.983946,-1.926980,0.316187,0.029038,0.065113
0,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.05,3000.0,True,0.05,102,0.084667,0.065144,0.103381,-2.481162,-2.977498,-1.905894,0.312590,0.025610,0.059868


In [4]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

disorder_sweep.to_csv(
    out_dir / "phase2_qrc_fixed_disorder_probe.csv",
    index=False,
)

print("Saved disorder probe to", out_dir)

Saved disorder probe to results/tables


In [5]:
# Stronger ridge on best fixed-disorder setting:
# 6q / PCA-6 / ZXZZ / snapshots / disorder_strength=0.20

strong_alpha_rows = []
strong_alpha_results = {}

for alpha in [3000, 10000, 30000]:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode="zxzz",
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=float(alpha),
        target_transform="log",
        seed=42,
        collect_anchor_features=True,
        use_disorder=True,
        disorder_strength=0.20,
    )

    run_name = f"6q_pca6_zxzz_snapshots_disorder020_alpha_{alpha}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    strong_alpha_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    strong_alpha_rows.append(row)

strong_alpha_sweep = pd.DataFrame(strong_alpha_rows)

strong_alpha_sweep[
    [
        "run_name",
        "ridge_alpha",
        "use_disorder",
        "disorder_strength",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
].sort_values("test_rmse")

Running 6q_pca6_zxzz_snapshots_disorder020_alpha_3000
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running 6q_pca6_zxzz_snapshots_disorder020_alpha_10000
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2

,run_name,ridge_alpha,use_disorder,disorder_strength,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,6q_pca6_zxzz_snapshots_disorder020_alpha_3000,3000.0,True,0.2,102,0.084320,0.063489,0.102618,-2.485080,-2.996562,-1.942716,0.319206,0.039238,0.072134
1,6q_pca6_zxzz_snapshots_disorder020_alpha_10000,10000.0,True,0.2,102,0.087293,0.063540,0.103204,-2.439361,-3.007861,-1.942674,0.291923,0.033753,0.064472
2,6q_pca6_zxzz_snapshots_disorder020_alpha_30000,30000.0,True,0.2,102,0.091056,0.063622,0.104600,-2.365533,-3.017457,-1.933111,0.262436,0.027507,0.053367


In [6]:
# Extend disorder sweep around the current best setting:
# 6q / PCA-6 / ZXZZ / snapshots / alpha=3000

more_disorder_rows = []
more_disorder_results = {}

for strength in [0.20, 0.30, 0.40, 0.50]:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode="zxzz",
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        collect_anchor_features=True,
        use_disorder=True,
        disorder_strength=float(strength),
    )

    run_name = f"6q_pca6_zxzz_snapshots_alpha3000_disorder_{strength}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    more_disorder_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    more_disorder_rows.append(row)

more_disorder_sweep = pd.DataFrame(more_disorder_rows)

more_disorder_sweep[
    [
        "run_name",
        "ridge_alpha",
        "use_disorder",
        "disorder_strength",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
].sort_values("test_rmse")

Running 6q_pca6_zxzz_snapshots_alpha3000_disorder_0.2
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running 6q_pca6_zxzz_snapshots_alpha3000_disorder_0.3
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 27

,run_name,ridge_alpha,use_disorder,disorder_strength,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.2,3000.0,True,0.2,102,0.084320,0.063489,0.102618,-2.485080,-2.996562,-1.942716,0.319206,0.039238,0.072134
1,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.3,3000.0,True,0.3,102,0.084580,0.062920,0.102771,-2.478748,-3.001398,-1.927606,0.315442,0.045710,0.070811
2,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.4,3000.0,True,0.4,102,0.084994,0.062830,0.103136,-2.468802,-3.000005,-1.903732,0.309052,0.046604,0.067208
3,6q_pca6_zxzz_snapshots_alpha3000_disorder_0.5,3000.0,True,0.5,102,0.085425,0.062886,0.103494,-2.458821,-2.997729,-1.880927,0.301858,0.045458,0.063793


In [7]:
# 8q / PCA-8 / ZXZZ / anchor snapshots + fixed disorder
# Mirrors the useful 6q upgrade path.

qrc8_disorder_rows = []
qrc8_disorder_results = {}

for strength in [0.05, 0.10, 0.20, 0.30]:
    config = TFIMQRCConfig(
        qubits=8,
        pca_components=8,
        lookback_days=40,
        anchor_count=8,
        anchor_policy="even",
        observable_mode="zxzz",
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        collect_anchor_features=True,
        use_disorder=True,
        disorder_strength=float(strength),
    )

    run_name = f"8q_pca8_zxzz_snapshots_alpha3000_disorder_{strength}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_8,
        config=config,
        target=target,
        verbose=True,
    )

    qrc8_disorder_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    qrc8_disorder_rows.append(row)

qrc8_disorder_sweep = pd.DataFrame(qrc8_disorder_rows)

qrc8_disorder_sweep[
    [
        "run_name",
        "ridge_alpha",
        "use_disorder",
        "disorder_strength",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
].sort_values("test_rmse")

Running 8q_pca8_zxzz_snapshots_alpha3000_disorder_0.05
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running 8q_pca8_zxzz_snapshots_alpha3000_disorder_0.1
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2

,run_name,ridge_alpha,use_disorder,disorder_strength,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
2,8q_pca8_zxzz_snapshots_alpha3000_disorder_0.2,3000.0,True,0.20,184,0.084501,0.063833,0.104041,-2.480406,-2.952330,-1.812051,0.315301,0.029667,0.061625
1,8q_pca8_zxzz_snapshots_alpha3000_disorder_0.1,3000.0,True,0.10,184,0.084651,0.064211,0.104172,-2.479364,-2.954270,-1.819851,0.313831,0.026741,0.057616
3,8q_pca8_zxzz_snapshots_alpha3000_disorder_0.3,3000.0,True,0.30,184,0.084386,0.063791,0.104173,-2.479966,-2.951178,-1.800625,0.316065,0.031289,0.061163
0,8q_pca8_zxzz_snapshots_alpha3000_disorder_0.05,3000.0,True,0.05,184,0.084754,0.064578,0.104357,-2.477572,-2.954445,-1.822315,0.312347,0.024878,0.053851


In [8]:
# PCA-5 probe:
# 6q / PCA-5 / ZXZZ / snapshots / disorder=0.20 / alpha=3000

pca5 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=5,
    prefix="pca5",
)

sequence_splits_5 = make_qrc_sequence_splits(
    pca5.splits,
    feature_columns=pca5.feature_columns,
    target_column=target,
    lookback_days=40,
)

print("PCA-5 cumulative variance:")
display(pca5.explained_variance)

pca5_config = TFIMQRCConfig(
    qubits=6,
    pca_components=5,
    lookback_days=40,
    anchor_count=6,
    anchor_policy="even",
    observable_mode="zxzz",
    trotter_steps_per_anchor=1,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    ridge_alpha=3000.0,
    target_transform="log",
    seed=42,
    collect_anchor_features=True,
    use_disorder=True,
    disorder_strength=0.20,
)

pca5_result = fit_tfim_qrc_regressor(
    sequence_splits_5,
    config=pca5_config,
    target=target,
    verbose=True,
)

pca5_summary = pd.DataFrame([summarize_qrc_result(pca5_result)])
pca5_summary[
    [
        "qubits",
        "pca_components",
        "anchor_count",
        "observable_mode",
        "collect_anchor_features",
        "use_disorder",
        "disorder_strength",
        "ridge_alpha",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
].T

PCA-5 cumulative variance:


,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.408573,0.408573
1,2,0.112421,0.520994
2,3,0.090849,0.611843
3,4,0.073259,0.685102
4,5,0.064206,0.749308


QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019


,0
qubits,6
pca_components,5
anchor_count,6
observable_mode,zxzz
collect_anchor_features,True
use_disorder,True
disorder_strength,0.2
ridge_alpha,3000.0
n_reservoir_features,102
train_rmse,0.085483
